In [1]:
"""
Test script to explore Stedin ArcGIS REST API responses
"""

import requests
import json
from pyproj import Transformer

# Bounding box from your config (Multatulibuurt, Delft)
bbox_coords = [
    (4.358862898190234, 51.989950476011565),
    (4.363513483963136, 51.99116041768696),
    (4.359959662710779, 51.997215138653104),
    (4.356868839817735, 51.996580034452485),
    (4.355168730042115, 51.995518297847504),
    (4.35819822166681, 51.9901601698577),
    (4.358862898190234, 51.989950476011565)
]

# Extract bounding box envelope in WGS84
lons = [coord[0] for coord in bbox_coords]
lats = [coord[1] for coord in bbox_coords]
xmin_wgs, xmax_wgs = min(lons), max(lons)
ymin_wgs, ymax_wgs = min(lats), max(lats)

print(f"Bounding box (WGS84): {xmin_wgs}, {ymin_wgs}, {xmax_wgs}, {ymax_wgs}")

# Transform to EPSG:28992 (Dutch coordinate system that Stedin uses)
transformer = Transformer.from_crs("EPSG:4326", "EPSG:28992", always_xy=True)
xmin_rd, ymin_rd = transformer.transform(xmin_wgs, ymin_wgs)
xmax_rd, ymax_rd = transformer.transform(xmax_wgs, ymax_wgs)

print(f"Bounding box (RD/EPSG:28992): {xmin_rd}, {ymin_rd}, {xmax_rd}, {ymax_rd}")
print("-" * 80)

# Actual Stedin Gas Network service URL (Layer 5)
gas_service_url = "https://services-eu1.arcgis.com/IQto421Ac9MzEmFT/arcgis/rest/services/KM_Gasvervangingsdata/FeatureServer/5"

print(f"Service: {gas_service_url}")
print("-" * 80)

# Build query parameters
params = {
    'where': '1=1',  # Get all features
    'geometry': f'{xmin_rd},{ymin_rd},{xmax_rd},{ymax_rd}',
    'geometryType': 'esriGeometryEnvelope',
    'inSR': '28992',  # Input spatial reference (Dutch RD)
    'spatialRel': 'esriSpatialRelIntersects',
    'outFields': '*',  # Get all attributes
    'returnGeometry': 'true',
    'f': 'geojson',  # Return as GeoJSON (easier to read than pbf)
    'outSR': '4326',  # Output as WGS84 for compatibility
    'resultRecordCount': 100  # Limit for testing
}

print("Query parameters:")
for key, value in params.items():
    print(f"  {key}: {value}")
print("-" * 80)

# Make the request
query_url = f"{gas_service_url}/query"
print(f"Request URL: {query_url}")
print("-" * 80)

try:
    response = requests.get(query_url, params=params, timeout=30)
    print(f"Status Code: {response.status_code}")
    print("-" * 80)
    
    if response.status_code == 200:
        data = response.json()
        
        # Print response structure
        print("Response type:", data.get('type'))
        print("Response keys:", list(data.keys()))
        print("-" * 80)
        
        if 'features' in data:
            print(f"✓ Number of features: {len(data['features'])}")
            print("-" * 80)
            
            if len(data['features']) > 0:
                print("First feature (full):")
                print(json.dumps(data['features'][0], indent=2))
                print("-" * 80)
                
                print("Available attributes in first feature:")
                if 'properties' in data['features'][0]:
                    for key, value in data['features'][0]['properties'].items():
                        print(f"  {key}: {value}")
                print("-" * 80)
                
                print("Geometry type:", data['features'][0].get('geometry', {}).get('type'))
                print("Geometry coordinates (first few):", str(data['features'][0].get('geometry', {}).get('coordinates'))[:200])
            else:
                print("⚠ No features returned in bounding box")
        
        if 'error' in data:
            print("✗ Error in response:")
            print(json.dumps(data['error'], indent=2))
    else:
        print(f"✗ Error {response.status_code}: {response.text[:500]}")
        
except requests.exceptions.RequestException as e:
    print(f"✗ Request failed: {e}")

print("\n" + "=" * 80)
print("Found service layers:")
print(f"  Gas Network: {gas_service_url}")
print("\nNext steps:")
print("1. Check if there are other layers (try /FeatureServer/0, /1, /2, etc.)")
print("2. Look for electricity network in the same service or different service")
print("3. Increase resultRecordCount if you need more features (max usually 2000)")
print("=" * 80)

Bounding box (WGS84): 4.355168730042115, 51.989950476011565, 4.363513483963136, 51.997215138653104
Bounding box (RD/EPSG:28992): 84113.91535326741, 445121.041619941, 84698.44430601149, 445921.12442740006
--------------------------------------------------------------------------------
Service: https://services-eu1.arcgis.com/IQto421Ac9MzEmFT/arcgis/rest/services/KM_Gasvervangingsdata/FeatureServer/5
--------------------------------------------------------------------------------
Query parameters:
  where: 1=1
  geometry: 84113.91535326741,445121.041619941,84698.44430601149,445921.12442740006
  geometryType: esriGeometryEnvelope
  inSR: 28992
  spatialRel: esriSpatialRelIntersects
  outFields: *
  returnGeometry: true
  f: geojson
  outSR: 4326
  resultRecordCount: 100
--------------------------------------------------------------------------------
Request URL: https://services-eu1.arcgis.com/IQto421Ac9MzEmFT/arcgis/rest/services/KM_Gasvervangingsdata/FeatureServer/5/query
--------------

In [7]:
# Explore available layers in the Stedin FeatureServer
base_service = "https://services-eu1.arcgis.com/IQto421Ac9MzEmFT/arcgis/rest/services/KM_Gasvervangingsdata/FeatureServer"

# Get service metadata
metadata_url = f"{base_service}?f=json"
response = requests.get(metadata_url, timeout=30)

if response.status_code == 200:
    service_info = response.json()
    
    print("Service Name:", service_info.get('serviceDescription', 'N/A'))
    print("\n" + "="*80)
    print("AVAILABLE LAYERS:")
    print("="*80)
    
    if 'layers' in service_info:
        for layer in service_info['layers']:
            print(f"\nLayer {layer['id']}: {layer['name']}")
            print(f"  Geometry Type: {layer.get('geometryType', 'N/A')}")
            if 'description' in layer:
                print(f"  Description: {layer['description']}")
    
    if 'tables' in service_info:
        print("\n" + "="*80)
        print("TABLES (non-spatial):")
        print("="*80)
        for table in service_info['tables']:
            print(f"\nTable {table['id']}: {table['name']}")
else:
    print(f"Error: {response.status_code}")

Service Name: Gasvervangingsdata, verbindingen en stations voor gebruik als open data. Deze data wordt door Stedin gebaseerd op interne data. Vrij te gebruiken.

AVAILABLE LAYERS:

Layer 1: Gasvervangingsdata
  Geometry Type: esriGeometryPolyline

Layer 2: Laagspanningsverbindingen
  Geometry Type: esriGeometryPolyline

Layer 3: Middenspanningsverbindingen
  Geometry Type: esriGeometryPolyline

Layer 4: Hoogspanningsverbindingen
  Geometry Type: esriGeometryPolyline

Layer 5: Laagspanningsstations
  Geometry Type: esriGeometryPolygon

Layer 6: MiddenLaagspanningsstations
  Geometry Type: esriGeometryPolygon

Layer 7: Middenspanningsstations
  Geometry Type: esriGeometryPolygon

Layer 8: Hoogspanningsstations
  Geometry Type: esriGeometryPolygon

TABLES (non-spatial):


In [3]:
# Check what attributes are available across all features
if data['features']:
    all_keys = set()
    for feature in data['features']:
        if 'properties' in feature:
            all_keys.update(feature['properties'].keys())
    
    print(f"Available attribute fields ({len(all_keys)}):")
    for key in sorted(all_keys):
        print(f"  - {key}")
    
    print("\n" + "="*80)
    print("Sample values from first 3 features:")
    print("="*80)
    
    # Show first 3 features as a table
    sample_data = []
    for i, feature in enumerate(data['features'][:3]):
        props = feature.get('properties', {})
        props['feature_index'] = i
        props['geometry_type'] = feature.get('geometry', {}).get('type')
        sample_data.append(props)
    
    df = pd.DataFrame(sample_data)
    print(df.to_string())

Available attribute fields (3):
  - OBJECTID
  - Shape__Area
  - Shape__Length

Sample values from first 3 features:
   OBJECTID  Shape__Area  Shape__Length  feature_index geometry_type
0       202     0.304291       2.321114              0       Polygon
1      1005     0.230225       2.008914              1       Polygon
2      1398     0.598835       3.206270              2       Polygon


In [4]:
# Convert to GeoDataFrame for easier manipulation
import geopandas as gpd

gdf = gpd.GeoDataFrame.from_features(data['features'], crs='EPSG:4326')
print(f"✓ Created GeoDataFrame with {len(gdf)} features")
print(f"\nColumns: {list(gdf.columns)}")
print(f"\nGeometry types:\n{gdf.geometry.type.value_counts()}")
print(f"\nDataFrame info:")
print(gdf.info())
print(f"\nFirst few rows:")
gdf.head()

✓ Created GeoDataFrame with 62 features

Columns: ['geometry', 'OBJECTID', 'Shape__Area', 'Shape__Length', 'feature_index', 'geometry_type']

Geometry types:
Polygon    62
Name: count, dtype: int64

DataFrame info:
<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 62 entries, 0 to 61
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype   
---  ------         --------------  -----   
 0   geometry       62 non-null     geometry
 1   OBJECTID       62 non-null     int64   
 2   Shape__Area    62 non-null     float64 
 3   Shape__Length  62 non-null     float64 
 4   feature_index  3 non-null      float64 
 5   geometry_type  3 non-null      object  
dtypes: float64(3), geometry(1), int64(1), object(1)
memory usage: 3.0+ KB
None

First few rows:


,geometry,OBJECTID,Shape__Area,Shape__Length,feature_index,geometry_type
0,"POLYGON ((4.35789 51.99535, 4.3579 51.99535, 4...",202,0.304291,2.321114,0.0,Polygon
1,"POLYGON ((4.35881 51.994, 4.35881 51.99401, 4....",1005,0.230225,2.008914,1.0,Polygon
2,"POLYGON ((4.35972 51.99403, 4.35972 51.99402, ...",1398,0.598835,3.206270,2.0,Polygon
3,"POLYGON ((4.36026 51.99281, 4.36026 51.99282, ...",1824,0.268028,2.139747,NaN,NaN
4,"POLYGON ((4.36146 51.99229, 4.36147 51.99229, ...",3595,0.320068,2.527768,NaN,NaN


In [6]:
# Plot the gas network features on an interactive map
import folium

# Project to local CRS (Dutch RD) for accurate centroid calculation
gdf_projected = gdf.to_crs(epsg=28992)
centroid_projected = gdf_projected.geometry.centroid
# Convert centroid back to WGS84
centroid_wgs = centroid_projected.to_crs(epsg=4326)
center = [centroid_wgs.y.mean(), centroid_wgs.x.mean()]

# Create map
m = folium.Map(location=center, zoom_start=15, tiles="OpenStreetMap")

# Add each feature to the map
for idx, row in gdf.iterrows():
    # Create popup with feature attributes
    popup_text = f"<b>Feature {idx}</b><br>"
    for col in gdf.columns:
        if col != 'geometry':
            popup_text += f"{col}: {row[col]}<br>"
    
    # Add feature to map (orange for gas network)
    folium.GeoJson(
        row.geometry,
        popup=folium.Popup(popup_text, max_width=300),
        style_function=lambda x: {
            'color': '#ff5100',
            'weight': 3,
            'opacity': 0.8
        }
    ).add_to(m)

# Add the bounding box polygon for reference
bbox_polygon = folium.Polygon(
    locations=[(lat, lon) for lon, lat in bbox_coords],
    color='blue',
    fill=False,
    weight=2,
    opacity=0.5,
    popup='Search Area'
).add_to(m)

# Save the map to a file
m.save('test_gas_network_map.html')
print("✓ Map saved to test_gas_network_map.html")

✓ Map saved to test_gas_network_map.html
